In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from PIL import Image
from her2_pcr_pathology_fm.tiling import tile_slide

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

In [ ]:
tests = [("yale", "S14-01774", "S14-01774.svs"),
         ("impress", "110", "110_HE.svs")]

for cohort, pid, fname in tests:
    tile_slide(DATA / cohort / "slides" / fname,
               ROOT / "features/tiles" / cohort / f"{pid}.h5",
               ROOT / "results/qc_tiling" / cohort / f"{pid}.png")
    print(f"done: {cohort}/{pid}")

In [ ]:
import h5py
for cohort, pid, _ in tests:
    with h5py.File(ROOT / "features/tiles" / cohort / f"{pid}.h5") as f:
        print(cohort, pid, "tiles:", len(f["coords"]), dict(f.attrs))
    display(Image.open(ROOT / "results/qc_tiling" / cohort / f"{pid}.png"))

### 1.0 Summarize tile counts

In [ ]:
import pandas as pd
qc = pd.read_csv(ROOT / "results/qc_tiling/tiling_qc.csv", dtype={"patient_id": str})
qc.groupby(["cohort", "label"])["n_tiles"].describe()
 # smallest slides
qc.sort_values("n_tiles").head(10)

In [ ]:
# largest slides
qc.sort_values("n_tiles").tail(10) 

### 2. Look at QC images

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def show(rows, ncols=4):
    rows = list(rows.itertuples())
    fig, axes = plt.subplots(-(-len(rows) // ncols), ncols, figsize=(4 * ncols, 4 * -(-len(rows) // ncols)))
    for ax, r in zip(axes.flat, rows):
        ax.imshow(Image.open(ROOT / f"results/qc_tiling/{r.cohort}/{r.patient_id}.png"))
        ax.set_title(f"{r.cohort} {r.patient_id}  n={r.n_tiles}"); ax.axis("off")
    for ax in axes.flat[len(rows):]:
        ax.axis("off")

show(qc.nsmallest(4, "n_tiles"))

In [ ]:
show(qc.nlargest(4, "n_tiles"))

### 4. Check whether tile count predicts the label

In [ ]:
from sklearn.metrics import roc_auc_score
for c, g in qc.groupby("cohort"):
    print(c, roc_auc_score(g["label"], g["n_tiles"]))

In [ ]:
import numpy as np
import openslide
import pandas as pd
import matplotlib.pyplot as plt
from PIL import ImageDraw
from her2_pcr_pathology_fm.tiling import tissue_mask, tile_coords, TILE_PX, TARGET_MPP, MASK_PX_PER_TILE

manifest = pd.read_csv(DATA / "manifest.csv", dtype={"patient_id": str})

def compare(pid, ceils=(255, 25)):
    row = manifest[manifest.patient_id == pid].iloc[0]
    slide = openslide.OpenSlide(str(DATA / row.slide_path))
    mpp = float(slide.properties[openslide.PROPERTY_NAME_MPP_X])
    tile_l0 = round(TILE_PX * TARGET_MPP / mpp)
    fig, axes = plt.subplots(1, len(ceils), figsize=(8 * len(ceils), 8))
    for ax, c in zip(axes, ceils):
        mask, thresh, thumb = tissue_mask(slide, tile_l0, sat_ceil=c)
        coords, _ = tile_coords(mask, tile_l0)
        img = thumb.copy()
        draw = ImageDraw.Draw(img)
        k = MASK_PX_PER_TILE
        for x, y in coords:
            x0, y0 = x / tile_l0 * k, y / tile_l0 * k
            draw.rectangle([x0, y0, x0 + k - 1, y0 + k - 1], outline=(0, 200, 0))
        ax.imshow(img)
        ax.set_title(f"{pid}  cap={c}  thresh={thresh:.0f}  n={len(coords)}")
        ax.axis("off")
    slide.close()

for pid in ["062", "S14-01774", "110", "086"]:
    compare(pid)

In [ ]:
def sat_view(pid, vmax=60):
    row = manifest[manifest.patient_id == pid].iloc[0]
    slide = openslide.OpenSlide(str(DATA / row.slide_path))
    mpp = float(slide.properties[openslide.PROPERTY_NAME_MPP_X])
    tile_l0 = round(TILE_PX * TARGET_MPP / mpp)
    mask, thresh, thumb = tissue_mask(slide, tile_l0, sat_ceil=255)
    sat = np.asarray(thumb.convert("HSV"))[..., 1]
    slide.close()

    fig, axes = plt.subplots(1, 3, figsize=(24, 8))
    axes[0].imshow(thumb); axes[0].set_title(f"{pid} thumbnail")
    im = axes[1].imshow(sat, cmap="magma", vmin=0, vmax=vmax)
    axes[1].set_title(f"saturation (Otsu thresh={thresh:.0f})")
    fig.colorbar(im, ax=axes[1], fraction=0.046)
    axes[2].imshow(mask, cmap="gray"); axes[2].set_title("mask (white = tissue)")
    for ax in axes: ax.axis("off")

sat_view("110")